# 2.2 — Giảm chiều dữ liệu Hotel Booking Demand

PCA phân tích phương sai. LDA dùng nhãn train để tìm một trục phân biệt hai lớp.

### Chạy trên Google Colab

Chọn **Runtime → Run all** và cấp quyền kết nối Google Drive khi Colab hỏi. Notebook tìm `clean_bookings_unscaled.csv` trong My Drive và các Shared drives đã được gắn. Nếu tệp chỉ nằm trong mục **Shared with me**, hãy thêm lối tắt của thư mục vào **My Drive**. Hãy sửa `INPUT_PATH` bên dưới khi có nhiều bản CSV hoặc khi bạn muốn dùng tệp ZIP chứa CSV.

Kết quả trên Colab được lưu dưới `/content/hotel-section-02/results/`. Nếu chạy Jupyter từ checkout, đặt CSV vào `data/processed/` hoặc điền đường dẫn vào `INPUT_PATH`.


In [ ]:
from pathlib import Path
import subprocess
import sys
from urllib.request import urlretrieve

# Có thể điền đường dẫn tuyệt đối đến CSV hoặc ZIP, ví dụ:
# INPUT_PATH = "/content/drive/MyDrive/ThuMucDuAn/data/processed/clean_bookings_unscaled.csv"
INPUT_PATH = ""
CSV_NAME = "clean_bookings_unscaled.csv"
SCRIPT_URL = (
    "https://raw.githubusercontent.com/lntd-ning/"
    "NguyenThiDieuLinh_Nhomso24_HotelCancellationPredection/main/src/dim_reduction.py"
)

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is not None:
    drive_root = Path("/content/drive")
    drive.mount(str(drive_root))
    root = Path("/content/hotel-section-02")
    script = root / "src/dim_reduction.py"
    script.parent.mkdir(parents=True, exist_ok=True)
else:
    cwd = Path.cwd()
    root = cwd.parent if cwd.name == "notebooks" else cwd
    script = root / "src/dim_reduction.py"

if INPUT_PATH:
    input_path = Path(INPUT_PATH).expanduser()
elif drive is not None:
    locations = [drive_root / "MyDrive", drive_root / "Shareddrives"]
    matches = sorted({path for folder in locations if folder.is_dir()
                      for path in folder.rglob(CSV_NAME) if path.is_file()})
    if len(matches) > 1:
        choices = "\n".join(str(path) for path in matches)
        raise ValueError(f"Tìm thấy nhiều bản CSV. Dán một đường dẫn vào INPUT_PATH:\n{choices}")
    if not matches:
        raise FileNotFoundError(
            f"Chưa tìm thấy {CSV_NAME} trong Drive đã gắn. "
            "Thêm lối tắt vào My Drive hoặc dán đường dẫn CSV/ZIP vào INPUT_PATH."
        )
    input_path = matches[0]
else:
    input_path = root / "data/processed" / CSV_NAME

if not input_path.is_file():
    raise FileNotFoundError(f"Không tìm thấy {input_path}. Hãy sửa INPUT_PATH.")

if drive is not None:
    urlretrieve(SCRIPT_URL, script)

print("Dữ liệu sử dụng:", input_path)
subprocess.run([sys.executable, str(script), str(input_path.resolve())], cwd=root, check=True)


### Đọc các bảng

Chọn số trục PCA theo phương sai tích lũy. Bảng validation so sánh quan hệ của từng trục với nhãn; AUC dùng chiều đã xác định ở train.

In [ ]:
import pandas as pd
from IPython.display import Image, display

tables = root / "results/tables/02_dim_reduction"
variance = pd.read_csv(tables / "pca_variance.csv")
cutoffs = [variance["cumulative_ratio"].ge(level).idxmax() for level in (0.80, 0.90, 0.95)]
display(variance.iloc[sorted(set(range(4)).union(cutoffs))])
display(pd.read_csv(tables / "axis_validation.csv").round(4))

### Trực quan hóa

Mục này vẽ sáu cặp trong bốn trục PCA đầu và mật độ trên LD1. LDA có một chiều vì dữ liệu có hai lớp.

In [ ]:
figures = root / "results/figures/02_dim_reduction"
for name in ("pca_variance.png", "pca_pairs.png", "lda_distribution.png"):
    display(Image(filename=str(figures / name)))